## Imports

In [8]:
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv
import os

## Configuration

In [9]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key
google_key = os.getenv("GOOGLE_API_KEY")
google_cse = os.getenv("GOOGLE_CSE_ID")

## Tools

In [10]:
# Funciones importadas desde otras partes del proyecto
# TODO realizar los imports correctamente cuando las herramientas esten listas
from tools import create_rag_tool, create_web_search_tool

# Crea herramientas inyectando las variables
rag_tool = create_rag_tool(openai_api_key=openai_api_key)
web_search_tool = create_web_search_tool(
    google_api_key=google_key,
    google_cse_id=google_cse
)

tools = [
    Tool(
        name="RAGTool",
        func=rag_tool,
        description="Usa esta herramienta para responder preguntas basadas en los apuntes del curso. Úsala por defecto."
    ),
    Tool(
        name="WebSearchTool",
        func=web_search_tool,
        description="Usa esta herramienta si el usuario explícitamente pide buscar en internet o menciona fuentes externas."
    )
]

## Memory for Context

In [11]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

## Prompt Context

In [12]:
agent_prompt = """
Eres un asistente conversacional experto en Inteligencia Artificial. 
Tu trabajo es ayudar a los estudiantes a responder preguntas basadas en sus apuntes del curso del primer semestre 2025. 

- Usa la herramienta RAG para buscar en los apuntes.
- Usa la herramienta de búsqueda en internet **solo si el usuario lo solicita explícitamente** con frases como "busca en internet", "verifica en la web", etc.
- Mantén el contexto de las preguntas anteriores para responder con coherencia.
- Sé claro, conciso y evita responder "no sé" si hay información en los apuntes.

Comienza ahora.
"""


## Init Agent

In [13]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125", temperature=0)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)


## Test

In [14]:
pregunta = input("Usuario: ")
respuesta = ""

if "internet" in pregunta.lower():
    respuesta = agent.run("Usa WebSearchTool: " + pregunta)
else:
    respuesta = agent.run(pregunta)

print(f"Agente: {respuesta}")




> Entering new AgentExecutor chain...
```json
{
    "action": "WebSearchTool",
    "action_input": "qué es una red neuronal"
}
```
Observation: [{'content': 'Las redes neuronales prealimentadas procesan los datos en una dirección, desde el nodo de entrada hasta el nodo de salida. Todos los nodos de una capa están\xa0...\nFuente: https://aws.amazon.com/es/what-is/neural-network/', 'metadata': {'source': 'https://aws.amazon.com/es/what-is/neural-network/'}}, {'content': '¿Qué son las redes neuronales? Una red neuronal es un programa o modelo de machine learning que toma decisiones de forma similar al cerebro humano, al emplear\xa0...\nFuente: https://www.ibm.com/es-es/think/topics/neural-networks', 'metadata': {'source': 'https://www.ibm.com/es-es/think/topics/neural-networks'}}, {'content': 'Compuestas por nodos interconectados llamados neuronas, las redes neuronales organizan estas unidades en capas. Cada neurona recibe entradas de otras, las\xa0...\nFuente: https://cloud.google.com/